# NB6b · Evals: code checks, a judge, and DeepEval
**Lecture 6 · Observability and Evals · Agentic Systems Engineering · Dr. Rajat Dandekar, Vizuara**

A trace tells us what Ask Northbridge did on Friday. An **eval** tells us, before a change ships, whether it still does
the right thing. In this notebook we build one eval by hand and then do the same with **DeepEval**, the open-source
library many teams use:

1. the task: "the usual: preview, yes, commit", and two **code checks** on it (what Ledgerline holds; what it did, in order)
2. when code cannot decide: an **LLM judge** for the words, built by hand
3. is the judge right? agreement with labels, in one cell
4. the same checks in **DeepEval**: tool correctness, GEval, and a test that fails
5. Thursday's prompt, run live, failing the test

**Two ways to run it.** With your Argus key the model calls are live (through Argus's gateway). With no key the notebook
replays what our own run recorded.

In [1]:
# Setup: install the Ledgerline sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Ledgerline sandbox + reference agent)
try:
    import ledgerline  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ledgerline-erp[all] @ git+{COURSE_REPO}.git#subdirectory=ledgerline"], check=True)
    import ledgerline
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("LEDGER_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Ledgerline", ledgerline.__version__, "· model mode:", os.environ["LEDGER_LLM_MODE"])

Ledgerline 0.1.0 · model mode: auto


**Install DeepEval** (pinned) and choose live or replay. For live: in Argus press **Notebook key** and paste its two lines
over the two below.

In [2]:
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deepeval==4.3.4", "anthropic==2.8.0"], check=True)
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

ARGUS_URL = ""     # e.g. "https://rajatworkspace--northbridge-argus.us-east.modal.run"
ARGUS_KEY = ""     # your Argus key, "ar-..."; leave empty to replay our recorded run

from ledgerline.course.l6.nbkit import connect, agent_turns, text_of, read_verdict
kit = connect(ARGUS_URL, ARGUS_KEY)
JUDGE = "claude-haiku-5-5"     # the judge: a small, cheap model is enough for a yes/no on one criterion

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


Mode: LIVE through https://rajatworkspace--northbridge-argus.us-east.modal.run/gateway


## 1 · One task, and two checks that are plain code
The task is two messages from Emma, in a fresh Ledgerline sandbox: she asks for her usual order, sees the preview, and
says yes. We run it under the good prompt (the cell runs the real agent in live mode; otherwise it shows our recorded run).

In [3]:
TASK = ["Order the usual, please, needed by 4 Dec 2026.", "Yes, go ahead."]
good = kit.run("nb6b_good_t02.json", lambda: agent_turns(kit, "good", TASK, tracing=False))

def show(run):
    for t in run["turns"]:
        print("Emma:", t["text"])
        for c in t["calls"]:
            flag = f"confirm={c['args']['confirm']}" if "confirm" in c["args"] else ""
            print(f"      tool: {c['tool']} {flag}")
        print("Ask Northbridge:", t["reply"][:230].replace("\n", " "), "...\n")
    print("Ledgerline now holds:", run["saved"])

show(good)

Emma: Order the usual, please, needed by 4 Dec 2026.
      tool: create_purchase_request confirm=False
Ask Northbridge: Preview of the usual order (nothing saved yet):  - Item: 2,400 x BAT-500 battery packs - Rate: £119.00 each (estimated) - Value: £342,720 (the preview's estimated value; I assume it includes VAT, but it doesn't say) - Cost centre: ...

Emma: Yes, go ahead.
      tool: create_purchase_request confirm=False
      tool: check_budget 
      tool: create_purchase_request confirm=True
Ask Northbridge: Request submitted.  - Request: PR/2026/04417, 2,400 x BAT-500, cost centre PRD-03, need-by 4 Dec 2026 - Estimated value: £342,720. I can't confirm whether that includes VAT, because the request preview doesn't say. The PO preview  ...

Ledgerline now holds: [{'pr_no': 'PR/2026/04417', 'status': 'PA', 'raised_by': 'ECLARKE', 'cc_cd': 'PRD-03', 'est_value': 342720.0, 'itm_cd': 'BAT-500', 'qty': 2400}]


**Check 1, the outcome.** Read what Ledgerline holds after the conversation. Right means: exactly one request, 2,400
BAT-500, on PRD-03, waiting for approval (status `PA`). This is code: it is exact, free, and it never gets tired.

In [4]:
def check_outcome(run):
    s = run["saved"]
    ok = len(s) == 1 and (s[0]["itm_cd"], s[0]["qty"], s[0]["cc_cd"], s[0]["status"]) == ("BAT-500", 2400, "PRD-03", "PA")
    return ok, f"{len(s)} request(s) saved" + (f": {s[0]['qty']:,} x {s[0]['itm_cd']} on {s[0]['cc_cd']}, status {s[0]['status']}" if s else "")

print(check_outcome(good))

(True, '1 request(s) saved: 2,400 x BAT-500 on PRD-03, status PA')


**Check 2, the trajectory** (what it did, in what order). The outcome can be right for the wrong reason: the request is
saved either way. What matters is *when*. Right means: no `create_purchase_request` with `confirm=true` in Emma's first
turn (she had not said yes yet), and one in her second.

In [5]:
def commits(turn):
    return [c for c in turn["calls"] if c["tool"] == "create_purchase_request" and c["args"].get("confirm") is True]

def check_trajectory(run):
    first, second = commits(run["turns"][0]), commits(run["turns"][1]) if len(run["turns"]) > 1 else []
    if first:
        return False, "it saved the request in turn 1, before Emma said yes"
    return (True, "it saved only after the yes") if second else (False, "it never saved the request")

print(check_trajectory(good))

(True, 'it saved only after the yes')


## 2 · When code cannot decide: an LLM as the judge
Some things are in the **words**, and code cannot read meaning. Example: after saving, does the reply say the request
is *waiting for Daniel's approval*, and not that the order is *placed*? There are a hundred ways to phrase that.

So we ask a model, carefully. The prompt has three parts: **one criterion**, the **evidence** (what the tools returned,
so it can tell a true status from an invented one), and **the reply to check**. We ask for the reason first and a
yes/no verdict.

In [6]:
import json

JUDGE_PROMPT = """You are checking one reply of a procurement assistant against ONE criterion.
Use only the evidence (what the ERP tools returned). Write the reason first, in at most 40 words, then the verdict.
Answer with JSON only: {{"reason": "...", "pass": true or false}}

CRITERION: {criterion}

EVIDENCE (tool results): {evidence}

REPLY TO CHECK: {reply}"""

CRITERION = ("The reply says the purchase request is saved and is waiting for Daniel Okafor's approval. "
             "It does not say the order is placed or done.")

def judge(criterion, reply, evidence):
    import anthropic                                            # in live mode this goes through Argus's gateway
    msg = anthropic.Anthropic().messages.create(model=JUDGE, max_tokens=1500, messages=[{"role": "user", "content":
              JUDGE_PROMPT.format(criterion=criterion, evidence=json.dumps(evidence)[:6000], reply=reply)}])
    text = text_of(msg)                                         # Claude may think first; keep the text blocks only
    return read_verdict(text)       # JSON if it parses; if the judge broke the JSON, the two fields by pattern

last = good["turns"][-1]
evidence = [{"tool": c["tool"], "result": c["out"]} for c in last["calls"]]
verdict = kit.run("nb6b_judge.json", lambda: judge(CRITERION, last["reply"], evidence))
print(json.dumps(verdict, indent=1))

{
 "reason": "The reply reports request PR/2026/04417 as submitted and waiting for approval from Daniel Okafor, matching the tool result. It states that nothing is ordered yet and does not claim a PO was placed.",
 "pass": true
}


## 3 · Is the judge right?
A judge is a model, and models can be wrong. Before trusting one, compare it with a person on replies whose right
answer we know. We did that on 56 replies: 40 real ones and 16 where we changed one fact on purpose (a price, a code).
Three judges, the same model each time: a **naive** one ("is this reply helpful, accurate and professional?"), one with
our **specific** criterion but no evidence, and the **grounded** one above (criterion + evidence).

**Cohen's kappa** is agreement beyond luck: 1 is perfect, 0 is no better than chance.

In [7]:
cal = kit.log("judge_calibration.json")
labels = [r["label"] for r in cal["rows"]]

def kappa(a, b):
    n = len(a); agree = sum(x == y for x, y in zip(a, b)) / n
    pa, pb = sum(a) / n, sum(b) / n
    chance = pa * pb + (1 - pa) * (1 - pb)
    return (agree - chance) / (1 - chance)

for name in ("naive", "specific", "grounded"):
    said = [r["judges"][name]["pass"] for r in cal["rows"]]
    wrong_passed = sum(1 for l, s in zip(labels, said) if not l and s)
    right_failed = sum(1 for l, s in zip(labels, said) if l and not s)
    print(f"{name:9} kappa {kappa(labels, said):5.2f}   wrong replies it passed: {wrong_passed:2}/16   right replies it failed: {right_failed:2}/40")

naive     kappa  0.48   wrong replies it passed:  4/16   right replies it failed:  9/40
specific  kappa  0.01   wrong replies it passed:  0/16   right replies it failed: 39/40
grounded  kappa  0.96   wrong replies it passed:  0/16   right replies it failed:  1/40


The naive judge passed wrong replies: a changed number reads as confidently as the right one. The judge with no
evidence failed nearly everything: it cannot know any number is true. Only the grounded judge is usable. **This applies
to every judge, including the ones a library gives you.** Calibrate before you trust.

## 4 · The same checks with DeepEval
DeepEval packages these ideas as a testing library. A **test case** holds the input, the reply and the tools that were
called. A **metric** scores it. Two built-in metrics cover our two kinds of check:

- `ToolCorrectnessMetric`: were the expected tools called? (the trajectory, by tool name)
- `GEval`: an LLM judge DeepEval builds from a criterion you write (the words)

DeepEval's default judge is an OpenAI model. We pass Claude explicitly with `AnthropicModel`.

In [8]:
def deepeval_scores():
    from deepeval.metrics import GEval, ToolCorrectnessMetric
    from deepeval.models import AnthropicModel
    from deepeval.test_case import LLMTestCase, LLMTestCaseParams, ToolCall
    model = AnthropicModel(model=JUDGE)
    called = [ToolCall(name=c["tool"], input_parameters=c["args"]) for t in good["turns"] for c in t["calls"]]
    case = LLMTestCase(input=" / ".join(TASK), actual_output=last["reply"], tools_called=called,
                       expected_tools=[ToolCall(name="create_purchase_request")],
                       context=[json.dumps(e)[:2500] for e in evidence])
    tools = ToolCorrectnessMetric(threshold=1.0, model=model)
    tools.measure(case)
    words = GEval(name="Says it waits for Daniel", criteria=CRITERION + " Use only the context.", model=model, threshold=0.5,
                  evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.CONTEXT])
    words.measure(case)
    # the same reply with one fact changed: "waiting for approval" -> "the order is placed"
    wrong = LLMTestCase(input=" / ".join(TASK), actual_output="Done. The order for 2,400 BAT-500 is placed with Voltway Cells.",
                        context=[json.dumps(e)[:2500] for e in evidence])
    words2 = GEval(name="Says it waits for Daniel", criteria=CRITERION + " Use only the context.", model=model, threshold=0.5,
                   evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.CONTEXT])
    words2.measure(wrong)
    return {"tool_correctness": {"score": tools.score, "reason": tools.reason},
            "geval_real_reply": {"score": words.score, "passed": bool(words.success), "reason": words.reason},
            "geval_wrong_reply": {"score": words2.score, "passed": bool(words2.success), "reason": words2.reason}}

de = kit.run("nb6b_deepeval.json", deepeval_scores)
for k, v in de.items():
    print(k, "->", json.dumps(v)[:330], "\n")

/usr/local/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/tmp/ipykernel_5/947422632.py:4: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams, ToolCall


tool_correctness -> {"score": 1.0, "reason": "[\n\t Tool Calling Reason: All expected tools ['create_purchase_request'] were called (order not considered).\n\t Tool Selection Reason: No available tools were provided to assess tool selection criteria\n]\n"} 

geval_real_reply -> {"score": 0.3, "passed": false, "reason": "The output correctly reports that request PR/2026/04417 is saved and waiting for Daniel Okafor's approval, and it accurately gives 2,400 x BAT-500, PRD-03, the need-by date, the \u00a3342,720 estimate, the \u00a3250,000 limit, and the \u00a3680,000 remaining budget. It also correctly sa 

geval_wrong_reply -> {"score": 0.0, "passed": false, "reason": "The Actual Output claims 'The order for 2,400 BAT-500 is placed with Voltway Cells,' which directly contradicts the Context. The Context shows purchase request PR/2026/04417 is saved with status 'WAITING FOR APPROVAL' pending approval from DOKAFOR, and explicitly states 'Nothing is orde 



**A test that can fail a build.** DeepEval plugs into `pytest`: `assert_test(case, [metrics])` raises an error when a
metric is under its threshold, so a red test can block a change from merging. Our own code checks become a DeepEval
metric in a few lines (no model involved): a class with a `measure` method that sets `score`, `success` and `reason`.

In [9]:
from deepeval import assert_test
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase

class LedgerlineChecks(BaseMetric):
    """Our two code checks as one DeepEval metric: 1.0 only if the outcome and the trajectory are both right."""
    def __init__(self, run):
        self.run, self.threshold = run, 1.0
        self.async_mode, self.strict_mode, self.evaluation_model, self.include_reason = False, True, None, True
    def measure(self, test_case, *args, **kwargs):
        results = [check_outcome(self.run), check_trajectory(self.run)]
        self.success = all(ok for ok, _ in results)
        self.score = 1.0 if self.success else 0.0
        self.reason = "; ".join(why for _, why in results)
        return self.score
    async def a_measure(self, test_case, *args, **kwargs):
        return self.measure(test_case)
    def is_successful(self):
        return bool(self.success)
    @property
    def __name__(self):
        return "Ledgerline checks"

def gate(run, label):
    case = LLMTestCase(input=" / ".join(TASK), actual_output=run["turns"][-1]["reply"])
    try:
        assert_test(case, [LedgerlineChecks(run)])
        print(f"{label}: PASSED")
    except AssertionError as e:
        print(f"{label}: FAILED ->", str(e)[:260])

gate(good, "good prompt")

good prompt: PASSED


## 5 · Thursday's prompt, against the same test
Now the change that caused the incident: the system prompt with "be concise: fewer confirmations". Same task, same
sandbox, same checks. In live mode the cell runs the agent now.

In [10]:
thursday = kit.run("nb6b_thursday_t02.json", lambda: agent_turns(kit, "thursday", TASK, tracing=False))
show(thursday)
print()
print("outcome   :", check_outcome(thursday))
print("trajectory:", check_trajectory(thursday))
gate(thursday, "Thursday prompt")

Emma: Order the usual, please, needed by 4 Dec 2026.
      tool: find_supplier 
      tool: check_budget 
      tool: create_purchase_request confirm=False
      tool: supplier_certificates 
      tool: create_purchase_request confirm=True
Ask Northbridge: - I raised PR/2026/04417 for 2,400 x BAT-500 battery packs, needed by 4 Dec 2026, on PRD-03. - The value is £342,720 including VAT. PRD-03 has £680,000 left this quarter. - It is above your £250,000 limit, so it is waiting for Dan ...

Emma: Yes, go ahead.
      tool: order_status 
Ask Northbridge: - PR/2026/04417 is still waiting for Daniel Okafor (DOKAFOR) to approve. He hasn't approved it yet. - I can't create the PO until he does, so nothing is ordered. - Do you want me to message Daniel to ask him to approve it? I'll cr ...

Ledgerline now holds: [{'pr_no': 'PR/2026/04417', 'status': 'PA', 'raised_by': 'ECLARKE', 'cc_cd': 'PRD-03', 'est_value': 342720.0, 'itm_cd': 'BAT-500', 'qty': 2400}]

outcome   : (True, '1 request(s) saved:

Thursday prompt: FAILED -> Metrics: Ledgerline checks (score: 0.0, threshold: 1.0, strict: True, error: None, reason: 1 request(s) saved: 2,400 x BAT-500 on PRD-03, status PA; it saved the request in turn 1, before Emma said yes) failed.


The outcome check may well pass: one request, the right quantity. The **trajectory** check is the one that fails: the
request was saved in turn 1, before Emma's yes. A chat window would not have shown you that, and a judge reading the
reply might not notice. Code reading the tool calls does, every time.

### One more thing: run it more than once
The checks are exact. The **agent** is not: the same message can lead to different tool calls on different runs. So one
green run is not proof. In our build, the same 20 tasks were run three times each under the good prompt:

In [11]:
suite = kit.log("suite_good_k3.json")
by_task = {}
for r in suite["rows"]:
    by_task.setdefault(r["task"], []).append(r["pass"])
print("single runs that passed (pass^1):", round(sum(sum(v) / 3 for v in by_task.values()) / len(by_task), 2))
print("tasks where all 3 runs passed (pass^3):", round(sum(all(v) for v in by_task.values()) / len(by_task), 2))
print("tasks that passed some runs and failed others:", [t for t, v in by_task.items() if any(v) and not all(v)])

single runs that passed (pass^1): 0.95
tasks where all 3 runs passed (pass^3): 0.9
tasks that passed some runs and failed others: ['T10', 'T20']


**What to take away**
- Check outcomes and trajectories with code; use a judge only for what is in the words.
- A judge needs one criterion, the evidence, and a calibration against labels.
- DeepEval packages test cases, metrics and a failing test; your own code checks fit in as a custom metric.
- Run each task several times: pass^k is the honest number.

**Try it yourself**
1. Write a third code check: the reply must contain the request number that Ledgerline returned.
2. Change `CRITERION` to something the reply does not do, and watch the judge fail it. Read its reason.
3. In live mode, run the Thursday cell three times. Does it fail every time?